In [2]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256,10))

X = torch.rand(2, 20)

In [3]:
net(X)

tensor([[ 0.1586,  0.1595,  0.0601,  0.0780, -0.0812, -0.0375, -0.2238,  0.0074,
         -0.0200,  0.0943],
        [ 0.1053,  0.1953,  0.0388,  0.0661, -0.0988,  0.0056, -0.1082, -0.0862,
          0.0348,  0.0031]], grad_fn=<AddmmBackward0>)

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))
    

In [5]:
net = MLP()
net(X)

tensor([[-0.0935, -0.0915, -0.1381, -0.0162, -0.0355, -0.1196, -0.0737,  0.1849,
         -0.0532,  0.0192],
        [-0.2457, -0.1753, -0.1158,  0.0123,  0.1015, -0.1225, -0.0498,  0.1029,
         -0.0095,  0.0120]], grad_fn=<AddmmBackward0>)

In [8]:
class Mysequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

In [9]:
net = Mysequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256,10))
net(X)

tensor([[ 0.0258,  0.0806,  0.1191,  0.0211,  0.0058,  0.1599, -0.0903, -0.0890,
         -0.0518, -0.1261],
        [ 0.0069,  0.0976,  0.2062, -0.0050, -0.0193,  0.1774, -0.0488, -0.0506,
         -0.0610, -0.1895]], grad_fn=<AddmmBackward0>)

In [16]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()
    

In [17]:
net = FixedHiddenMLP()
net(X)

tensor(-0.0272, grad_fn=<SumBackward0>)